In [ ]:
import leafmap
file_path = "./Unprocessed/clipped_LC08_L2SP_027040_20200412_20200822_02_T1_ST_B10.TIF"
m = leafmap.Map(center=getLongitudeLatitudeOfTif(file_path), zoom=6)
m.add_raster(file_path, layer_name="Raster Layer")
m

In [1]:
import os
import shutil

def clear_folder(folder_path):
    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.remove(file_path)
                print(f"Deleted file: {file_path}")
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
                print(f"Deleted folder: {file_path}")
        except Exception as e:
            print(f"Failed to delete {file_path}. Reason: {e}")

# Call the function
clear_folder('./RawClippedRasters')


Deleted folder: ./RawClippedRasters\Albedo
Deleted folder: ./RawClippedRasters\DEM
Deleted folder: ./RawClippedRasters\Land_Cover
Deleted folder: ./RawClippedRasters\LST
Deleted folder: ./RawClippedRasters\NDVI
Deleted folder: ./RawClippedRasters\NDWI


In [ ]:
# Define search payload (if filtering is needed, adjust accordingly)
dataset_search_payload = {}

# Send request to dataset search endpoint
datasets = sendRequest(serviceUrl + "dataset-search", dataset_search_payload, apiKey)

# Print the names of available datasets
for ds in datasets:
    # print(ds.keys())
    print(f"Dataset Name: {ds['datasetCategoryName']} + -> {ds['datasetAlias']}")

In [13]:
from pynlcd import get_land_cover
from osgeo import ogr
import os

# Define file paths
shapefile_folder = "./Data/area_shp/"
shapefile = "Polygon_San_Antonio_TX.shp"
shapefile_path = shapefile_folder + shapefile

# Load shapefile using GDAL/OGR
driver = ogr.GetDriverByName('ESRI Shapefile')
dataSource = driver.Open(shapefile_path, 0)  # 0 means read-only, 1 means writable
if dataSource is None:
    raise FileNotFoundError(f"Shapefile not found at {shapefile_path}")

layer = dataSource.GetLayer()
feature = layer.GetNextFeature()
if feature is None:
    raise ValueError("No features found in shapefile.")

# Extract ROI geometry
roi_geom = feature.GetGeometryRef()

# Define extent from geometry bounds
min_x, max_x, min_y, max_y = roi_geom.GetEnvelope()
extent = (min_x, max_x, min_y, max_y)

# Download the land cover data
output_path = './Unprocessed/'
get_land_cover(roi_geom, extent, year=2001, spatial_resolution=0.0003, output_path=output_path)
print(f"Land cover data saved to {output_path}")

Getting the image...
Image saved to ./Unprocessed/NLCD_2001_Land_Cover.tif
Land cover data saved to ./Unprocessed/


In [ ]:
centroid = aoi_geodf.geometry.centroid.iloc[0]
m = folium.Map(
    location=[centroid.y, centroid.x], 
    zoom_start=9, tiles="openstreetmap", width="100%", height="100%", attributionControl=0
)
# Cell 4: Add Polygon to Map
folium.GeoJson(aoi_geodf).add_to(m)
m